# Ma Soi — RL (PPO from the BC clone) locally

Runs the RL experiment on this machine, one stage per run, resumable. Same run folders
and flags as the Colab/Kaggle notebooks (spec
`docs/superpowers/specs/2026-09-17-rl-ppo-from-bc-design.md`, v3/D11), so their results
mix: e.g. run `village` on Kaggle, copy its `rl/` folder into `.tmp/rl/`, continue here.

The logic lives in `ai-training/rl_stages.py` (tested in `ai-training/tests/test_rl_stages.py`);
this notebook only sets the stage and shows the output.

## How to use

1. Set `STAGE` in the next cell (start with `"village"`).
2. Run all cells. The last line prints `>>> NEXT:` — the stage to run next.
3. Order: `village` → (`village-lr3` if needed) → `wolves` → (`wolves-lr3`) → `night`
   (optional) → `confirm`. Send the `VERDICT` block of `confirm` to Claude.

| Stage | This machine |
|---|---|
| `village`, `wolves` (20 iterations each) | ~4 h |
| `night` (10 iterations × 2 sides) | ~4 h |
| `confirm` | ~1 h |

## Shut down / interrupted?

Run the SAME cells again with the same `STAGE`. Finished iterations and finished steps
are skipped; only the step that was running restarts (at most ~8 min of rollout or
~17 min of benchmark lost). **Interrupt** (■) also stops `rl_loop` and its node processes.

While a stage runs, Windows is kept awake (the screen may still turn off). Keep VS Code
open; closing it stops the kernel.

In [ ]:
PROJECT = "b"          # a = spec 2026-09-17 | b = spec 2026-09-19 (vote history)
STAGE = "confirm"      # village | village-lr3 | wolves | wolves-lr3 | night | confirm
BUDGET_HOURS = None    # e.g. 3.0 = stop cleanly after 3 h; re-run later to continue
RL_DIR = None          # None = <repo>/.tmp/rl (shared with rl_stages.py in a terminal)
CONFIRM_MODEL = None   # confirm only: a specific model path instead of the auto-picked champion

## 0. Environment

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai-training" / "rl_stages.py").exists()), None)
if ROOT is None:
    raise SystemExit(f"cannot find the repo root from {Path.cwd()} - open this notebook inside the repo")
VENV_PY = ROOT / "ai-training" / ".venv" / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
if not VENV_PY.exists():
    raise SystemExit(f"missing {VENV_PY} - install the venv per ai-training/README.md")

sys.path.insert(0, str(ROOT / "ai-training"))
import importlib
import rl_stages as rs
importlib.reload(rs)          # pick up edits without restarting the kernel
rs.PYTHON = VENV_PY           # rl_loop/train_ppo need torch, which lives in the venv
rs.set_project(PROJECT)
RL = Path(RL_DIR).resolve() if RL_DIR else ROOT / ".tmp" / "rl"

# Rollouts call the TypeScript engine directly: build it once per session.
npm = "npm.cmd" if sys.platform == "win32" else "npm"
done = subprocess.run([npm, "run", "build:deps", "--silent"], cwd=ROOT, capture_output=True, text=True)
if done.returncode != 0:
    print(done.stdout[-2000:], done.stderr[-2000:])
    raise SystemExit("npm run build:deps failed")
print("repo ", ROOT)
print("venv ", VENV_PY)
print("runs ", RL)
print("stage", STAGE, "| budget", f"{BUDGET_HOURS} h" if BUDGET_HOURS else "none")

## 1. Status (what is already done)

In [ ]:
print("\n>>> suggested stage:", rs.status(RL))

## 2. Run the stage

Long-running (hours). Output streams below and into `.tmp/rl/<run>.log`.

In [ ]:
if not rs.CHAMPION0.exists():
    raise SystemExit(f"missing {rs.CHAMPION0}")
t0 = time.time()
deadline = t0 + BUDGET_HOURS * 3600 if BUDGET_HOURS else None
with rs.KeepAwake():
    hint = rs.run_stage(STAGE, RL, deadline, Path(CONFIRM_MODEL) if CONFIRM_MODEL else None)
print("\n>>> NEXT:", hint)
print(f">>> elapsed {(time.time() - t0) / 3600:.1f} h")